# 11. Talking to a Board — Serial (pyserial)

> We send and receive **one line of text at a time** with a board plugged in over USB (ESP32 and others). Use it to move LEDs · motors from AI results, or to receive and log sensor values.

### What you will learn
1. Serial basics: port · speed (baud) · end of line (EOL)
2. Open · send · receive · close with pyserial
3. Making a simple protocol: ignore the echo, pick the answer by its prefix
4. Getting numbers out of a received line

### First — safety
- If the [Board] app in edge-lab is using the port, you cannot open it here — disconnect in the Board app or close edge-lab.
- Connect the board by USB only. **Give high-current parts such as servos · motors their own power supply, not power from USB**
  (if several move at once, the voltage drops and the board restarts). Connect the GND of the external supply and the board together.
- ESP32 GPIO pins are **3.3V**. Do not connect a 5V sensor output directly (use a voltage divider · level shifter).

### Terms
| Term | Meaning |
|---|---|
| Port | Windows `COM3`, Ubuntu `/dev/ttyUSB0` · `/dev/ttyACM0` |
| baud | Bits sent per second. **Both sides must match** (usually 115200) |
| End of line (EOL) | The mark that a line is finished. `\n` or `\r\n` — match the firmware |
| DTR / RTS | Control lines. Many USB-UART boards are **reset** through these lines |

## 1. Find the port

In [ ]:
import os
import re
import time
import serial
import serial.tools.list_ports

ports = list(serial.tools.list_ports.comports())
for p in ports:
    print(f"{p.device:14s} {p.description}  (VID {p.vid:04X})" if p.vid else f"{p.device:14s} {p.description}")
PORT = os.environ.get("BOARD_PORT") or (ports[0].device if ports else None)
print("Port to use:", PORT, "← if it is wrong, write it yourself: PORT = 'COM5'")

The VID tells you the chip: `303A` Espressif (ESP32 built-in USB) · `10C4` CP210x · `1A86` CH340 · `0403` FTDI · `2E8A` RP2040.

## 2. Open
If you open the port with DTR/RTS **kept low**, the board does not reset the moment you open it (CP210x · CH340 boards). Whether this also works on the ESP32-S3 built-in USB depends on the board — check it.
`timeout=0.2` — the longest time (seconds) `readline()` waits for a line. Without it, it waits forever when no line comes.

In [ ]:
BAUD = 115200
EOL = "\n"                                   # end of line — change it if the firmware expects \r\n

ser = serial.Serial()
ser.port, ser.baudrate, ser.timeout = PORT, BAUD, 0.2
ser.dtr = ser.rts = False
ser.open()
time.sleep(0.3)
ser.reset_input_buffer()                     # throw away lines that piled up before opening
print("Opened:", ser.name)

## 3. Send · send and get an answer
- Text is sent as **bytes**: `"PING\n".encode("utf-8")`
- Turn received bytes back into text: `.decode("utf-8", errors="replace")` → so broken characters do not stop the program
- Some boards send the command straight back (echo) — a line equal to what you sent does not count as the answer
- If the board keeps streaming sensor values, such a line can be caught as the answer → pick by **prefix**

In [ ]:
def send(text):
    ser.write((text + EOL).encode("utf-8"))

def read_line():
    return ser.readline().decode("utf-8", errors="replace").strip()

def ask(text, wait=1.0, prefix=""):
    """Send, then return the first line that comes in afterwards (starting with prefix). None if there is none."""
    ser.reset_input_buffer()
    send(text)
    end = time.time() + wait
    while time.time() < end:
        line = read_line()
        if line and line != text and line.startswith(prefix):
            return line
    return None

print("Answer:", ask("PING"))
print("Answer (only lines starting with PO):", ask("PING", prefix="PO"))

## 4. Read lines the board sends by itself — get the numbers
To get just the number from a line like `T=23.5`, a regular expression is handy: `re.search(r"T=([-\d.]+)", line)`.

In [ ]:
def number_of(line, key):
    m = re.search(rf"{key}=([-\d.]+)", line)
    return float(m.group(1)) if m else None

end = time.time() + 5
while time.time() < end:
    line = read_line()
    if line:
        print(f"{time.strftime('%H:%M:%S')}  {line:20s} → T value: {number_of(line, 'T')}")

## Common errors
| Symptom | Cause · fix |
|---|---|
| `PermissionError` / `could not open port` | Another program (Board app · Arduino IDE Serial Monitor) is using it. On Ubuntu, join the `dialout` group |
| Garbled characters | The baud rate is different |
| No answer comes | End of line (EOL) does not match the firmware · the firmware does not know that command |
| Board restarts every time you open the port | DTR/RTS reset → open with them kept low. If it still happens, it may be the board circuit — check it |

## Summary
- Serial = both sides agree on port · baud · end of line, then send and receive text one line at a time as bytes.
- Picking the answer: ignore the echo + use a prefix. Always put a time limit on waiting.
- Close the port when you are done — so other programs can use it.

---
# 🛠 Mini project — Sensor logger + remote LED

**What to build**: For 20 seconds, collect the sensor values the board sends (`T=…`), save them to a **CSV file**, and draw a **graph**.
When the value goes above the limit (`LIMIT`), send `LED ON`; when it goes back down, send `LED OFF` (send only when it changes).

**What you use**: `read_line` · `number_of` · `send` (from this notebook), `csv`, `matplotlib`

The command names (`LED ON` etc.) are set by the firmware — change them to match your firmware.

In [ ]:
import csv
import matplotlib.pyplot as plt     # keep chart text in English

LIMIT = 25.0
SECONDS = 20
rows, led = [], None
t0 = time.time()
while time.time() - t0 < SECONDS:
    line = read_line()
    v = number_of(line, "T") if line else None
    if v is None:
        continue
    rows.append((round(time.time() - t0, 2), v))
    want = v > LIMIT
    if want != led:                                   # send only when it changes
        send("LED ON" if want else "LED OFF")
        led = want
        print(f"{rows[-1][0]:5.1f}s  T={v} → {'LED ON' if want else 'LED OFF'}")

with open("sensor_log.csv", "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerows([("sec", "T")] + rows)
print(f"Saved {len(rows)} rows: sensor_log.csv")

if rows:
    xs, ys = zip(*rows)
    plt.figure(figsize=(7, 2.5)); plt.plot(xs, ys, marker=".")
    plt.axhline(LIMIT, color="r", ls="--"); plt.xlabel("seconds"); plt.ylabel("T"); plt.show()

In [ ]:
ser.close()                                   # always close it when you are done
print("Closed")

**Try more**
- Send the finger count from lesson 06 to the board and light that many LEDs (`send(f"LED {n}")`)
- Instead of turning on as soon as the value crosses the limit once, turn on only after 3 crossings in a row (to stop flickering)
- Open the CSV in Excel